In [1]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_ibm import WatsonxEmbeddings
from langchain_ibm import ChatWatsonx

# from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain.agents import create_agent

from langchain_core.tools import Tool
from langchain_core.tools import tool

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda

/Users/sunahgwak/Documents/Repository/source/ollama/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

apiKey = os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watsonx_ai_url = os.getenv("WATSONX_URL")
hf_token = os.environ["HF_TOKEN"]
COHERE_API_KEY = os.environ["COHERE_API_KEY"]
SERPER_API_KEY = os.getenv("SERPER_API_KEY")

watson_embedding = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual",
    url = f"{watsonx_ai_url}",
    api_key = f"{apiKey}",
    project_id=f"{project_id}"
)

watson_llm = ChatWatsonx(
  model_id="ibm/granite-4-h-small",
  url=f"{watsonx_ai_url}",
  api_key = f"{apiKey}",
  project_id=f"{project_id}",
  max_tokens = 2000,
  params = {
    "temperature":0
  }
)

ollama_embedding = OllamaEmbeddings(model="nomic-embed-text-v2-moe")

hugging_llm = ChatOpenAI(
  base_url = "https://router.huggingface.co/v1",
  api_key=hf_token,
  model="Qwen/Qwen2.5-7B-Instruct:together",
  temperature=0
)

qwen_llm = ChatOllama(model="qwen3.5:4b", temperature=0)


#### RouterChain(LCEL Router: 조건부 분기 패턴)
- Router 패턴: 입력이 무엇인가에 따라서 적절한 체인 or Runnable로 분기
- 입력 분석/ 분류기가 입력을 분석(LLM기반 or 규칙기반) -> 적절한 체인으로 라우팅
- |
- ex) 질문 유형(코딩/수학/일반...)에 따라 다른 전문 프롬프트를 적용
- 동작 흐름
  (1) 입력 => (2) 분류 => (3) 라우팅 => (4) 실행 => (5) 반환

In [5]:
parser = StrOutputParser()

# 수학체인
math_chain = ChatPromptTemplate.from_messages([
  ("system", "당신은 수학 전문가입니다. 풀이 과정을 단계별로 설명하세요"),
  ("human", "{question}")  
]) | watson_llm | parser

# 코드체인
code_chain = ChatPromptTemplate.from_messages([
  ("system", "당신은 시니어 개발자입니다. 고드와 주석을 함께 제공하세요"),
  ("human", "{question}")  
]) | watson_llm | parser

# 일반체인
general_chain = ChatPromptTemplate.from_messages([
  ("system", "당신은 친절한 AI 어시스턴트입니다. 한국어로 답해주세요"),
  ("human", "{question}")  
]) | watson_llm | parser

# 질문을 읽고 유형 반환
classify_chain = ChatPromptTemplate.from_messages([
  ("system", "질문 유형을 math/code/general 중 하나로만 답하세요"),
  ("human", "{question}")  
]) | watson_llm | parser

# 라우터함수
def route(inputs:dict):
  category = classify_chain.invoke(inputs).strip().lower()
  print(f" => 분류 결과 {category}")
  
  if 'math' in category: return math_chain
  elif 'code' in category: return code_chain
  else: return general_chain

# 둘이 같은 의미
# router_chain = RunnableLambda(route) | RunnableLambda(lambda chain: chain)
router_chain = RunnableLambda(lambda x:route(x).invoke(x))

In [8]:
questions=[
  {"question": "피타고라스 정리를 증명해줘"},
  {"question": "오늘 저녁 메뉴 추천해줘"},
  {"question": "파이썬으로 버블 정렬을 구현해주라"}
]

for q in questions:
  print(f"Q: {q['question']}")
  print(f"a: {router_chain.invoke(q)[:10]}...\n")

Q: 피타고라스 정리를 증명해줘
 => 분류 결과 피타고라스 정리는 직각삼각형에서 빗변의 길이의 제곱이 나머지 두 변의 길이의 제곱의 합과 같다는 정리입니다. 수식으로 표현하면 다음과 같습니다:

a^2 + b^2 = c^2

여기서 c는 빗변의 길이이고, a와 b는 나머지 두 변의 길이입니다.

피타고라스 정리의 증명은 여러 가지가 있지만, 여기서는 가장 간단한 증명 중 하나를 소개하겠습니다.

증명:

1. 직각삼각형 abc를 그립니다. 여기서 각 a는 직각이고, 빗변은 bc입니다.
2. 삼각형 abc의 세 변에 각각 정사각형을 그립니다. 변 ab에는 정사각형 abde, 변 ac에는 정사각형 acfg, 변 bc에는 정사각형 bhij를 그립니다.
3. 정사각형 abde와 acfg를 변 ab와 ac에 대해 각각 반시계 방향으로 90도 회전시킵니다. 이때, 정사각형 abde는 정사각형 bhij와 합쳐져서 정사각형 bdeh를 이룹니다. 정사각형 acfg는 정사각형 acfg와 합쳐져서 정사각형 acfg'를 이룹니다.
4. 이제 두 개의 큰 정사각형 bdeh와 acfg'가 생겼습니다. 이 두 정사각형의 넓이를 비교해보겠습니다.
5. 정사각형 bdeh의 넓이는 (ab + ac)^2입니다. 이는 변 ab와 ac의 길이를 더한 값의 제곱이기 때문입니다.
6. 정사각형 acfg'의 넓이는 bc^2입니다. 이는 빗변 bc의 길이의 제곱이기 때문입니다.
7. 두 정사각형의 넓이를 비교해보면, (ab + ac)^2 = bc^2입니다.
8. 식을 풀면, ab^2 + 2ab\*ac + ac^2 = bc^2가 됩니다.
9. 이때, 직각삼각형에서 두 변의 길이의 곱은 0이므로, 2ab\*ac = 0입니다.
10. 따라서, ab^2 + ac^2 = bc^2가 됩니다.

이로써 피타고라스 정리가 증명되었습니다.
a: 피타고라스 정리는 직각삼각형에서 빗변의 길이의 제곱이 나머지 두 변의 길이의 제곱의 합과 같다는 정리입니다. 수식으로 표현하면 다음과 같습니다:

a^2 + b^2 = c^2


#### RunnableBranch(선언형 분기)

In [9]:
from langchain_core.runnables import RunnableBranch

# RunnableBranch((조건1,체인1), (조건2,체인2))
branch = RunnableBranch(
  (lambda x:'math' in x.get('topic', ''), math_chain),
  (lambda x:'code' in x.get('topic', ''), code_chain),
  (lambda x:'cooking' in x.get('topic', ''), ChatPromptTemplate.from_messages([
      ("system", "당신은 요리 전문가입니다"),
      ("human", "question")
    ]) | watson_llm | parser),
  general_chain
)

print(branch.invoke({'topic':'math', 'question': '미분이란?'}))
print(branch.invoke({'topic':'cooking', 'question': '김치찌개레시피'}))
print(branch.invoke({'topic':'other', 'question': '안녕하세요'}))
print(branch.invoke({'topic':'code', 'question': '파이썬으로 선택정렬 구현'}))

미분은 미적분학의 한 분야로, 함수의 변화율을 나타내는 수학적 개념입니다. 미분을 통해 함수의 기울기, 즉 함수가 특정 지점에서 얼마나 빠르게 증가하거나 감소하는지를 알 수 있습니다. 이를 통해 함수의 최대값, 최소값, 곡률 등을 분석할 수 있습니다.

미분의 기본 개념은 함수의 한 점에서의 접선의 기울기를 구하는 것입니다. 이를 위해 미분은 한 점에서의 함수의 변화율을 극한으로 정의합니다. 함수 f(x)의 x=a에서의 미분은 다음과 같이 정의됩니다:

f'(a) = lim(h→0) [(f(a+h) - f(a))/h]

여기서 f'(a)는 함수 f(x)의 x=a에서의 미분계수, 즉 x=a에서의 함수의 기울기를 나타냅니다. 이 식은 h가 0에 가까워질 때, (a, f(a))와 (a+h, f(a+h)) 사이의 선분의 기울기가 함수 f(x)의 x=a에서의 기울기에 수렴한다는 것을 의미합니다.

미분의 과정은 다음과 같은 단계로 이루어집니다:

1. 함수 f(x)를 주어진다.
2. 함수 f(x)에 대해 미분 공식을 적용하여 미분 계수 f'(x)를 구한다. 이 과정을 미분이라고 합니다.
3. 필요한 경우, 특정 지점 x=a에서의 미분계수 f'(a)를 구하여 그 지점에서의 기울기를 알아낸다.

미분은 다양한 응용 분야에서 사용되며, 물리학, 공학, 경제학 등에서 중요한 역할을 합니다. 예를 들어, 물리학에서는 물체의 속도와 가속도를 구하기 위해 미분을 사용하고, 경제학에서는 수요와 공급 함수의 변화율을 분석하기 위해 미분을 사용합니다.
물론입니다! 요리에 대해 궁금한 점이 있으시다면 언제든지 질문해 주세요. 저는 요리에 대한 다양한 정보와 팁을 제공해 드리겠습니다. 어떤 주제에 대해 알고 싶으신가요?
안녕하세요! 도움이 필요하시면 언제든지 말씀해 주세요. 어떤 도움이 필요하신가요?
물론입니다. 선택 정렬(Selection Sort)은 주어진 리스트에서 최소값을 찾아서 맨 앞부터 차례대로 정렬하는 알고리즘입니다. 파이썬으로 선택 정렬을 구현하는 코드와 주석을 함께 제공하겠

### SequentialChain(단계별 순차 파이프라인)
- 앞 단계 출력이 뒷 단계의 입력으로 흘러 들어감
- | => SequentialChain
- .assign(): 중간 결과를 보존하며 여러 단계를 누적할 수 있음
- 각 단계의 출력 타입과 입력 타입이 일치해야 함

In [10]:
# 체인 정의
translate_chain = ChatPromptTemplate.from_messages([
  ("system", "다음 텍스트를 한국어로 번역하세요. 번역문만 출력:\n{text}")
]) | watson_llm | parser

summarize_chain = ChatPromptTemplate.from_messages([
  ("system", "다음 텍스트를 3문장으로 번역하세요:\n{text}")
]) | watson_llm | parser

sentiment_chain = ChatPromptTemplate.from_messages([
  ("system", "다음 텍스트의 감정을 긍정/부정/중립 중 하나로만 대답하세요:\n{summary}")
]) | watson_llm | parser

report_chain = ChatPromptTemplate.from_messages([
  ("system", "아래 분석 결과를 바탕으로 한 줄 최종 보고서를 작성하세요:\n"
   "원문: {text}\n번역: {translated}\n요약:{summary}\n감정:{sentiment}")
]) | watson_llm | parser

# assign(): 단계별 결과 누적
pipeline = (RunnablePassthrough
            .assign(translated=translate_chain)
            .assign(summary=summarize_chain)
            .assign(sentiment=sentiment_chain)
            .assign(report=report_chain)
            )

# 결과만 다음으로 넘어간다. 과정이 누적이 안된다.
# assign 사용한 이유 -> 순차적으로 흘러 가면서 흐름이 남는다. 단계별 흔적 누적
result = pipeline.invoke({"text":"Python is a versatile language loved by developers worldwide."})
print("번역: ", result['translated'])
print("요약: ", result['summary'])
print("김정: ", result['sentiment'])
print("보고서: ", result['report'])

번역:  파이썬은 개발자들에게 사랑받는 다용도의 언어입니다.
요약:  파이썬은 개발자들에게 사랑받는 다재다능한 언어입니다.
김정:  긍정
보고서:  파이썬은 개발자들에게 사랑받는 다재다능한 언어입니다.


In [12]:
# 조입부 단계 삽입

detect_lang_chain = ChatPromptTemplate.from_messages([
  ("system","다음 텍스트의 언어를 korean/english/other중 하나로만 대답하세요: \n{text}")
]) | watson_llm | parser

# 조건에 따라 추가
def maybe_translate(inputs):
  lang = detect_lang_chain.invoke(inputs).strip().lower()
  
  if 'english' in lang or 'other' in lang:
    translated = translate_chain.invoke(inputs)
    return {**inputs, "text":translated, 'was_translated':True}
  return {**inputs, 'was_translated':False}

smart_pipeline = (RunnableLambda(maybe_translate) | RunnablePassthrough.assign(summary = summarize_chain) | RunnablePassthrough.assign(sentiment = sentiment_chain))

r1 = smart_pipeline.invoke({'text':'Python is Great!'})
r2 = smart_pipeline.invoke({'text':'파이썬은 최고야'})

print(r1['was_translated'], r1["summary"][:30])
print(r2['was_translated'], r2["summary"][:30])

True Here is the translation in 3 s
False Here is the translation in 3 s


### MapReduceChain(대용량 문서 처리)
- Map: 문서를 청크로 나눠 각각 처리(요약, 추출..) => 병렬 실행 가능
- Reduce: Map 결과들을 하나로 합쳐 최종 답변 생성

In [15]:
# 문서 형태 pdf
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

map_chain = ChatPromptTemplate.from_messages([
  ("system", "다음 텍스트를 2문장으로 요약하세요"),
  ("user", "{chunk}")
]) | watson_llm | parser

reduce_chain = ChatPromptTemplate.from_messages([
  ("system", "다음은 긴 문서의 섹션별 요약입니다. 전체를 5문장으로 통합 요약하세요"),
  ("user", "{summaries}")
]) | watson_llm | parser

# Map: 문서를 청크로 나눔
def map_reduce_summarize(file_path):
  loader = PyPDFLoader(file_path)
  splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
  chunks = splitter.split_documents(loader.load())
  print(f"총 청크 수 {len(chunks)}")
  # ---- Map: 청크별 요약
  chunk_inputs = [{'chunk':c.page_content} for c in chunks]
  summaries = map_chain.batch(chunk_inputs, config={'max_concurrency':5})
  print(f"총 청크 수 {len(summaries)} 개 요약 생성")
  
  # ---- Reduce: 요약 통합
  # 문서 결합
  combined = "\n\n".join(f"[색션] {i+1} {s}" for i, s in enumerate(summaries))
  final = reduce_chain.invoke({'summaries':combined })
  return final
  
  
result = map_reduce_summarize("./data/2026 상 삼성E&A 직무기술서.pdf")
print(result)


총 청크 수 41
총 청크 수 41 개 요약 생성
삼성E&A는 2026년 대학생 인턴 모집에서 기술직(설계/조달)과 기술직(사업관리/시공관리/품질) 두 가지 직군을 운영하며, 각 직군은 프로젝트의 설계부터 종료까지 다양한 역할을 수행합니다. 기술직(설계/조달)은 설계와 상세설계, 주요 기자재 구매, 공정관리 등을 담당하고, 기술직(사업관리/시공관리/품질)은 프로젝트 전 과정을 계획하고 관리하며, 품질 보증과 관리를 수행합니다. 삼성E&A는 화공플랜트, 석유화학, 바이오 의약품 플랜트, 재생에너지와 수전해 기술을 활용한 그린 수소 솔루션 등 다양한 사업을 수행하며, 지속 가능한 미래를 위한 다양한 솔루션을 제공합니다. 신입사원들은 각 분야에서 전문성을 키우며, 글로벌 엔지니어들과의 협업을 통해 프로젝트의 성공적인 수행을 도모합니다.
